Given that the RT-IoT2022 dataset is about detecting network attacks, and such attacks often manifest as anomalous, outlier-like behavior, RobustScaler is often a strong candidate. It will scale the data without being overly influenced by the extreme values that could be indicative of attacks. This is the reason that I’ll be using a RobustScaler for Scaling.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Identify categorical columns in X
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

if not categorical_cols.empty:
    # Initialize OneHotEncoder
    # handle_unknown='ignore' will allow the encoder to ignore categories not seen during fit
    # sparse_output=False ensures a dense array output
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # Fit and transform the categorical columns
    encoded_features = encoder.fit_transform(X[categorical_cols])

    # Create a DataFrame from the encoded features
    encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
    encoded_df = pd.DataFrame(encoded_features, columns=encoded_feature_names, index=X.index)

    # Drop the original categorical columns from X
    X_numerical = X.drop(columns=categorical_cols)

    # Concatenate the numerical part of X with the new encoded features
    X = pd.concat([X_numerical, encoded_df], axis=1)

    print("X after One-Hot Encoding categorical features:")
    display(X.head())
    print(f"New shape of X: {X.shape}")
else:
    print("No categorical features found in X to encode.")

Before encoding, based on X DataFrame it had two categorical columns: 'proto' and 'service'. These columns contained text values (e.g., 'tcp', 'udp' for 'proto'; 'mqtt', 'dns' for 'service'). Most machine learning algorithms require numerical input, so we need to convert these text categories into numbers. After One-Hot Encoding:

•	**New Encoded Columns:**

You now see new columns like proto_icmp, proto_tcp, proto_udp, service_-, service_dhcp, service_dns, etc. For example, if a row originally had 'tcp' in the 'proto' column, the proto_tcp column for that row will be 1.0, and proto_icmp, proto_udp will be 0.0.

•**	Increased Column Count:**

The print statement New shape of X: (117915, 94) shows that the number of columns in X has increased from 83 to 94. This increase (94 - 83 = 11 new columns) corresponds to the total number of unique categories across the 'proto' and 'service' columns.
In essence, I’ve transformed the categorical information into a numerical format that preserves the distinctness of each category, making the X DataFrame ready for most machine learning models.


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Instantiate LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the 'Attack_type' column of y
y['Attack_type'] = label_encoder.fit_transform(y['Attack_type'])

print("y DataFrame after Label Encoding:")
display(y.head())

print("\nUnique values in 'Attack_type' column after encoding:")
print(y['Attack_type'].unique())

print("\nMapping of original labels to encoded integers:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label}: {i}")

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Ensure X contains only numerical features for mutual_info_classif
# Categorical features in X should have already been one-hot encoded.
# For mutual_info_classif, we need to ensure all columns in X are numeric.
# The 'flow_duration_binned' is already numerical (integer labels).
# The one-hot encoded columns are also numerical (0.0 or 1.0).

# Convert y['Attack_type'] to a 1D array as required by mutual_info_classif
y_encoded = y['Attack_type'].values

# Calculate mutual information scores
mi_scores = mutual_info_classif(X, y_encoded, random_state=42)

# Create a Series for better readability and indexing
mi_series = pd.Series(mi_scores, index=X.columns)

# Sort the features by mutual information score in descending order
mi_series_sorted = mi_series.sort_values(ascending=False)

print("Mutual Information Scores (Top 20 Features):")
display(mi_series_sorted.head(20))

Reasoning: To better understand the relative importance of the features, visualizing the top 20 mutual information scores is crucial. A bar plot will effectively represent these scores.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the top 20 mutual information scores
plt.figure(figsize=(12, 8))
sns.barplot(x=mi_series_sorted.head(20).values, y=mi_series_sorted.head(20).index, hue=mi_series_sorted.head(20).index, palette='viridis', legend=False)
plt.title('Top 20 Features by Mutual Information Score')
plt.xlabel('Mutual Information Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Select Top Features

### Subtask:
Based on the mutual information scores, select a subset of features (e.g., the top N features or features above a certain threshold) for further modeling.


**Reasoning**:
To comply with the instruction to select the top N features, I will define N as 20, extract the column names of these top features from the previously calculated `mi_series_sorted`, and then update the `X` DataFrame to retain only these selected features.



In [ ]:
n_top_features = 20

# Get the names of the top N features
top_features_names = mi_series_sorted.head(n_top_features).index.tolist()

# Update X to include only these selected features
X = X[top_features_names]

print(f"X DataFrame updated to include only the top {n_top_features} features.")
print("Selected features:")
display(X.head())
print(f"New shape of X: {X.shape}")

Based on this, we can see the distribution of the new flow_duration_binned column in the output, which shows the count of data points falling into each of the 5 bins. A bar plot further visualizes this distribution.
The flow_duration_binned column is now part of our X DataFrame, as shown in the head of the DataFrame. This new feature can be used in your models, potentially helping to capture non-linear relationships or simplify the feature's representation.


The new domain-specific features based on ratios and aggregations have been successfully added to the X DataFrame. We can see the updated X DataFrame (its first few rows) and the new shape, which now has 104 columns. These new features are designed to capture more nuanced patterns in the network traffic data, which can be very beneficial for improving the performance and interpretability of your machine learning models

Now that the target variable y is encoded, the next step is to calculate the mutual information score between each feature in X and the encoded target variable y. This will help quantify the dependency between each feature and the attack type.
To better understand the relative importance of the features, visualizing the top 20 mutual information scores is crucial. A bar plot will effectively represent these scores.


In [ ]:
print("Top 20 Features and their Mutual Information Scores:")
display(mi_series_sorted.head(20))

print("\nFinal X DataFrame with selected features:")
display(X.head())
print(f"New shape of X: {X.shape}")

In the filter method, I applied Mutual Information as suggested, which measures the dependency between two random variables. Here, it evaluates how much information a specific feature (X column) provides about the target variable (Attack_type). A higher score means a stronger relationship, indicating that the feature is more useful for predicting attack types. Unlike correlation, which only identifies linear relationships, Mutual Information can reveal both linear and non-linear connections between variables.